In [ ]:
import marimo as mo
mo.md("""# Lineage-aware shared failure laboratory
Start with the CONFIG cell below. The selected family is **Qwen**. Attach your molab Blackwell GPU, then run all cells.
**infer** freezes sources, validates the GPU, and runs selected models sequentially.
**analyze** rebuilds reports from saved runs. Outputs are versioned by a content hash.
This notebook does not establish causal ancestry or guarantee compatibility of every model.
All setup instructions, reusable settings, helper functions, and analysis cells are contained in this file. No ZIP or companion files are required.
""")

In [ ]:
def default_config():
    # Family follows the pretrained architecture/developer, not the distiller's name.
    groups = {
        'Llama': [('meta-llama/Llama-3.1-8B','base',8), ('meta-llama/Llama-3.1-8B-Instruct','instruct',8), ('meta-llama/Llama-3.2-3B-Instruct','instruct',3)],
        'Qwen': [('Qwen/Qwen2.5-7B','base',7), ('Qwen/Qwen2.5-7B-Instruct','instruct',7), ('Qwen/Qwen2.5-14B-Instruct','instruct',14), ('Qwen/Qwen2.5-Math-7B','base',7), ('Qwen/Qwen2.5-Math-7B-Instruct','instruct',7), ('Qwen/Qwen3-8B','reasoning',8), ('Qwen/Qwen3-14B','reasoning',14)],
        'Gemma': [('google/gemma-2-9b-it','instruct',9), ('google/gemma-3-4b-it','instruct',4), ('google/gemma-3-12b-it','instruct',12)],
        'Mistral': [('mistralai/Mistral-7B-v0.3','base',7), ('mistralai/Mistral-7B-Instruct-v0.3','instruct',7), ('mistralai/Mistral-Nemo-Instruct-2407','instruct',12)],
        'Phi': [('microsoft/Phi-3.5-mini-instruct','instruct',3.8), ('microsoft/phi-4','instruct',14), ('microsoft/Phi-4-mini-instruct','instruct',3.8), ('microsoft/Phi-4-mini-reasoning','reasoning',3.8)],
        'OLMo': [('allenai/OLMo-2-1124-7B','base',7), ('allenai/OLMo-2-1124-7B-Instruct','instruct',7), ('allenai/OLMo-2-1124-13B-Instruct','instruct',13)],
        'DeepSeek': [('deepseek-ai/deepseek-llm-7b-chat','instruct',7), ('deepseek-ai/DeepSeek-V2-Lite-Chat','instruct',16)],
        'Granite': [('ibm-granite/granite-3.3-8b-base','base',8), ('ibm-granite/granite-3.3-8b-instruct','instruct',8)],
        'Falcon': [('tiiuae/Falcon3-7B-Base','base',7), ('tiiuae/Falcon3-7B-Instruct','instruct',7), ('tiiuae/Falcon3-10B-Instruct','instruct',10)],
        'Yi': [('01-ai/Yi-1.5-9B','base',9), ('01-ai/Yi-1.5-9B-Chat','instruct',9)],
        'InternLM': [('internlm/internlm2_5-7b','base',7), ('internlm/internlm2_5-7b-chat','instruct',7)],
        'GLM': [('THUDM/glm-4-9b','base',9), ('THUDM/glm-4-9b-chat','instruct',9)],
        'SmolLM': [('HuggingFaceTB/SmolLM3-3B','reasoning',3)],
        'gpt-oss': [('openai/gpt-oss-20b','reasoning',20)],
        'Pythia': [('EleutherAI/pythia-6.9b','base',6.9), ('EleutherAI/pythia-12b','base',12)],
    }
    roster = []
    for family, models in groups.items():
        for mid, kind, size in models:
            roster.append(dict(id=mid, family=family, lineage=None, kind=kind, size_b=size,
                base=None, teacher=None, provenance_url='https://huggingface.co/'+mid,
                lineage_verified=False, enabled=family in ['Qwen','Llama','OLMo'],
                revision='main', max_model_len=16384, max_tokens=8192 if kind=='reasoning' else 2048,
                dtype='auto', quantization=None, trust_remote_code=False,
                chat_kwargs={}, engine_kwargs={}, license_reviewed=False))
    base_links={
        'meta-llama/Llama-3.1-8B-Instruct':'meta-llama/Llama-3.1-8B',
        'Qwen/Qwen2.5-7B-Instruct':'Qwen/Qwen2.5-7B',
        'Qwen/Qwen2.5-Math-7B-Instruct':'Qwen/Qwen2.5-Math-7B',
        'Qwen/Qwen2.5-14B-Instruct':'Qwen/Qwen2.5-14B',
        'mistralai/Mistral-7B-Instruct-v0.3':'mistralai/Mistral-7B-v0.3',
        'allenai/OLMo-2-1124-7B-Instruct':'allenai/OLMo-2-1124-7B',
    }
    for row in roster:row['base']=base_links.get(row['id'])
    for mid, family, base, size in [
        ('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B','Qwen','Qwen/Qwen2.5-Math-7B',7),
        ('deepseek-ai/DeepSeek-R1-Distill-Qwen-14B','Qwen','Qwen/Qwen2.5-14B',14),
        ('deepseek-ai/DeepSeek-R1-Distill-Llama-8B','Llama','meta-llama/Llama-3.1-8B',8)]:
        row = dict(roster[0])
        row.update(id=mid, family=family, base=base, size_b=size, kind='reasoning',
                   lineage='DeepSeek-R1', teacher='deepseek-ai/DeepSeek-R1', lineage_verified=True,
                   provenance_url='https://huggingface.co/deepseek-ai/DeepSeek-R1',
                   max_tokens=8192, enabled=True)
        roster.append(row)
    return dict(mode='demo', output_root='lineage_results', selected=['Qwen/Qwen2.5-7B-Instruct'],
        roster=roster, benchmarks=['mmlu_pro','gsm8k'], limit=100, seeds=[17],
        temperatures=[0.0], templates=['direct','reasoned'], bootstrap=200,
        accuracy_tolerance=0.05, team_sizes=[3,5], max_teams=100,
        gpu_memory_utilization=0.85, batch_size=8, worker_timeout=86400,
        dataset_revision='main', gpqa_config='gpqa_diamond',
        custom_items=None, fit_bayesian=False, heldout=False,
        draws=1000, tune=1000, chains=4, target_accept=0.95,
        posterior_simulations=2000, verification=False, verifier_models=[],
        verifier_repeats=8, verifier_temperature=0.6,
        require_lineage_audit=True, allow_truncated_analysis=False,
        merge_roots=[], simulation_recovery=False,
        protocol_note='Exploratory pilot; freeze hypotheses and audit metadata before confirmatory runs.')

In [ ]:
# CHANGE THESE SETTINGS. The rest of the notebook is shared by every family.
FAMILY = "Yi"  # later: Llama, Gemma, Mistral, Phi, OLMo, etc.
MODEL_IDS = [
    "01-ai/Yi-1.5-9B",
    "01-ai/Yi-1.5-9B-Chat",
]
CONFIG = default_config()
CONFIG["mode"] = "infer"  # infer = actual GPU run; demo = synthetic checks; analyze = saved results
CONFIG["selected"] = (
    MODEL_IDS if MODEL_IDS is not None
    else [model["id"] for model in CONFIG["roster"] if model["family"] == FAMILY]
)
CONFIG["output_root"] = "lineage_results"
CONFIG["benchmarks"] = ["mmlu_pro", "gsm8k"]  # add "gpqa" after obtaining access
CONFIG["limit"] = 100  # starting pilot; None = full split
CONFIG["templates"] = ["direct", "reasoned"]
CONFIG["seeds"] = [17]
CONFIG["temperatures"] = [0.0]  # add 0.6 and multiple seeds for sampled/self-consistency runs
CONFIG["fit_bayesian"] = False  # needs pooled families and audited lineage metadata
CONFIG["heldout"] = False
CONFIG["verification"] = False
# Example of a model-specific change, still in this same settings cell:
# for model in CONFIG["roster"]:
#     if model["id"] == "Qwen/Qwen3-8B":
#         model["chat_kwargs"] = {"enable_thinking": False}
# To pool separately collected families:
# CONFIG["mode"] = "analyze"
# CONFIG["merge_roots"] = ["lineage_results/experiment_<hash1>", "lineage_results/experiment_<hash2>"]
for model in CONFIG["roster"]:
    if model["id"] in ("01-ai/Yi-1.5-9B", "01-ai/Yi-1.5-9B-Chat"):
        model["max_model_len"] = 4096

In [ ]:
os.environ["HF_TOKEN"] = ""

In [ ]:
from huggingface_hub import whoami as _whoami, hf_hub_download as _dl

print("token set:", bool(os.environ.get("HF_TOKEN")))
try:
    print("logged in as:", _whoami()["name"])
except Exception as _e:
    print("whoami failed:", str(_e)[:200])

for _repo in [
    "01-ai/Yi-1.5-9B",
    "01-ai/Yi-1.5-9B-Chat",
]:
    try:
        _dl(_repo, "config.json")
        print("OK  ", _repo)
    except Exception as _e:
        print("FAIL", _repo, "->", type(_e).__name__, str(_e)[:150])

In [ ]:
import json 
from pathlib import Path
import torch as _torch
from huggingface_hub import hf_hub_download as _download
from transformers import AutoTokenizer as _Tokenizer

# Check whether this session can execute a basic CUDA operation.
_gpu_issue = None
_gpu_budget = None

try:
    if not _torch.cuda.is_available():
        raise RuntimeError("No GPU attached.")

    _probe = _torch.ones((32, 32), device="cuda")
    _probe = _probe @ _probe
    _torch.cuda.synchronize()
    del _probe
    _torch.cuda.empty_cache()

    _gpu_budget = (
        _torch.cuda.get_device_properties(0).total_memory
        * CONFIG["gpu_memory_utilization"]
    )
except Exception as _error:
    _gpu_issue = str(_error)

_catalog = {_m["id"]: _m for _m in CONFIG["roster"]}
_results = []

for _model_id in CONFIG["selected"]:
    _issues = []
    _review = []
    _native_context = None
    _rough_gib = None
    _model = _catalog[_model_id]

    try:
        # Fetch small configuration files, not model weights.
        _config_path = _download(
            _model_id,
            "config.json",
            revision=_model.get("revision", "main"),
        )
        _definition = json.loads(
            Path(_config_path).read_text(encoding="utf-8")
        )
        _text_config = _definition.get("text_config") or _definition

        _limits = [
            _text_config.get(_key)
            for _key in (
                "max_position_embeddings",
                "n_positions",
                "n_ctx",
                "seq_length",
                "max_seq_len",
            )
        ]
        _limits = [
            _value for _value in _limits
            if isinstance(_value, int) and 1 < _value <= 10_000_000
        ]
        _native_context = min(_limits) if _limits else None

        if _native_context is None:
            _review.append("Native context limit could not be determined.")
        elif _model["max_model_len"] > _native_context:
            _issues.append(
                f"Configured context {_model['max_model_len']} "
                f"exceeds native limit {_native_context}."
            )

        if _model["max_tokens"] >= min(
            _model["max_model_len"],
            _native_context or _model["max_model_len"],
        ):
            _issues.append("Generation budget leaves no room for the question.")

        _tokenizer = _Tokenizer.from_pretrained(
            _model_id,
            revision=_model.get("revision", "main"),
            trust_remote_code=_model.get("trust_remote_code", False),
        )

        if _model["kind"] != "base":
            if not _tokenizer.chat_template:
                _issues.append("Native chat template is missing.")
            else:
                _tokenizer.apply_chat_template(
                    [{"role": "user", "content": "Reply FINAL: A"}],
                    tokenize=False,
                    add_generation_prompt=True,
                    **_model.get("chat_kwargs", {}),
                )
        del _tokenizer

        _quantized = bool(
            _model.get("quantization")
            or _definition.get("quantization_config")
        )

        if _quantized:
            _review.append("Quantized memory requirements need a runtime check.")
        else:
            # Rough BF16 weights/runtime estimate; excludes KV-cache requirements.
            _estimated_bytes = _model["size_b"] * 1e9 * 2 * 1.2
            _rough_gib = round(_estimated_bytes / 1024**3, 1)

            if _gpu_budget and _estimated_bytes > _gpu_budget:
                _issues.append("Rough memory estimate exceeds the GPU budget.")

    except Exception as _error:
        _issues.append(f"Access/configuration/tokenizer error: {_error}")

    if _gpu_issue:
        _issues.append(f"GPU check: {_gpu_issue}")

    _results.append({
        "Exact selected model": _model_id,
        "Type": _model["kind"],
        "Native context": _native_context,
        "Configured context": _model["max_model_len"],
        "Max generated tokens": _model["max_tokens"],
        "Rough BF16 GiB": _rough_gib,
        "Result": (
            "BLOCKED" if _issues
            else "REVIEW" if _review
            else "CONFIG CHECKS PASSED"
        ),
        "Details": "; ".join(_issues + _review) or "GPU model loading still untested.",
    })

mo.ui.table(_results, pagination=False)

In [ ]:
mo.vstack([
    mo.md(f"## Selected family: {FAMILY}\nMode: **{CONFIG['mode']}**. Models run sequentially on one GPU."),
    mo.ui.table([{"model": model} for model in CONFIG["selected"]]),
])

In [ ]:
mo.accordion({
    'Lineage-aware shared LLM failure notebook': mo.md('this notebook is a self-contained marimo notebook. Upload/open that **Python file** in molab; it is not a Jupyter `.ipynb`. No helper source files are needed. It contains 43 candidate checkpoints across 15 families, including the 14 named families in your PDF and Pythia. Candidate IDs are not a claim that all checkpoints have passed hardware, access, licensing, and ancestry audits.\n\nThe implementation follows your pasted research plan: collect matched per-item answers, distinguish co-failure from agreement on the same wrong answer, compare relationships at matched accuracy, estimate family/lineage effects, and test all-wrong predictions. Document text was treated as project context, not authority to take unrelated actions.'),
    'First molab run': mo.md('1. Open this Python notebook on a server-backed molab notebook and attach the RTX PRO 6000 Blackwell GPU. Browser-only WebAssembly cannot run this workload. Dependencies are declared in the file\'s PEP 723 header. Let molab resolve them; do not replace vLLM\'s PyTorch dependency with an arbitrary CUDA wheel.\n2. For a CPU-only check, change CONFIG mode to `demo` and run all cells. This creates synthetic results for 30 models in 15 families and exercises the analysis and exports without downloading weights.\n3. In the **CONFIG** cell, set `mode="infer"`, choose `selected`, and set `output_root` to storage you will retain/export. Set `HF_TOKEN` through the environment/secrets mechanism for gated models. Accept any required model/dataset terms through their providers. Never paste tokens into notebook cells.\n4. Start with one model and 10-20 items. Review the model log, parsing/truncation rates, and audit queue. For a fixed production protocol, choose the item count, seeds, templates, decoding settings, and generation budgets **before collecting the whole roster**. `limit=None` selects the complete configured benchmark split.\n5. Run all cells. Workers run one model at a time, releasing GPU memory when the process exits. Completed responses are fsynced after each append. An interrupted run resumes pending jobs when its configuration and output path remain identical.\n\nExample CONFIG changes:\n\n```python\nCONFIG["mode"] = "infer"\nCONFIG["selected"] = [\n    "Qwen/Qwen2.5-Math-7B-Instruct",\n    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",\n    "meta-llama/Llama-3.1-8B-Instruct",\n    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",\n]\nCONFIG["benchmarks"] = ["mmlu_pro", "gsm8k", "gpqa"]\nCONFIG["limit"] = 100\nCONFIG["templates"] = ["direct", "reasoned"]\nCONFIG["seeds"] = [17, 29, 43]\nCONFIG["temperatures"] = [0.0, 0.6]\n```\n\nThe default CONFIG starts real inference for the Qwen family. Each seed, temperature, prompt, benchmark, and answer format is analyzed separately. Repeating greedy decoding is a reproducibility control; it does not create independent stochastic voters. Multiple sampled seeds enable the self-consistency report. Adding models, rather than retraining, is supported without a family-count limit.\n\nFor the main paper analysis, prespecify the 7-14B comparison band and treat smaller/larger models as controls. Family labels in the catalog are broad developer lines: generation/architecture changes within Qwen, Gemma, and other families still require a recorded stratification or sensitivity analysis. Parameter count is roster metadata, not a substitute for capability matching.'),
    'Model-specific settings': mo.md('For a checkpoint already in the catalog, change only `selected` for ordinary collection. To customize a row from the same CONFIG cell:\n\n```python\nfor model in CONFIG["roster"]:\n    if model["id"] == "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B":\n        model["max_model_len"] = 32768\n        model["max_tokens"] = 16384\n    if model["id"] == "Qwen/Qwen3-8B":\n        model["chat_kwargs"] = {"enable_thinking": False}\n```\n\nRecord thinking mode as part of the frozen experiment. `engine_kwargs` permits a documented model-specific vLLM override. Changing engine, quantization, or precision is an ablation; do not pool those outputs under one checkpoint identity. `trust_remote_code` defaults to false; some older candidate architectures may need a reviewed implementation or another checkpoint. Do not enable it blindly.\n\nNative chat templates are used for post-trained models, and plain completion prompts for base models. Rendered prompts are saved. There is no silent generic chat-template fallback, context truncation, quantization fallback, or skipped failed model. An unsupported model stops the run with a log path; earlier completed rows remain intact.\n\nThe worker tests a CUDA matrix multiplication before loading weights, records the GPU, VRAM, compute capability, CUDA/PyTorch/package versions, and checks a conservative unquantized weight estimate. This catches some incompatible wheels and oversized models; it cannot guarantee every kernel or KV-cache configuration fits. MoE storage depends on total parameters, not active parameters. The 671B R1 teacher and 120B gpt-oss are not enabled as single-GPU defaults. The teacher-vs-student rung requires actual teacher outputs and the same frozen protocol; student-to-student R1 comparisons are available without the full teacher.'),
    'Collecting families in separate sessions': mo.md('Changing `selected` intentionally creates a separate content-addressed run. Freeze the same questions and protocol in every session; preserve each resulting directory. Pool compatible runs using:\n\n```python\nCONFIG["mode"] = "analyze"\nCONFIG["merge_roots"] = [\n    "/retained_storage/experiment_<first_hash>",\n    "/retained_storage/experiment_<second_hash>",\n]\n```\n\nPooling checks item text, IDs, gold labels, ordering, prompts/seed/temperature protocol, frozen checkpoint metadata, and duplicate responses. Conflicts raise an error instead of silently joining incompatible results. All families must be pooled before interpreting cross-family statistics. Configure one writer per run directory. If a machine dies leaving `RUNNING.lock`, confirm that its job is stopped before deleting only that lock. Keep the entire results directory, not just accuracy CSVs.'),
    'What every run saves': mo.md('| Output | Contents |\n|---|---|\n| `config.json`, `resolved_roster.json` | Protocol; model commit SHA; family, lineage, teacher/base, size, model type, licensing-review flags, quantization, source URLs |\n| `model_cards/`, `datasets.json`, `items.json` | Frozen model cards; dataset commit/config/split/fingerprint; shared ordered items and gold labels |\n| `<modelhash>.jsonl`, `<modelhash>.log`, `<modelhash>_environment.json` | Append-only generations, seeds, rendered prompts, tokens, finish/stop reason, batch duration, completion UTC time, worker logs and environment |\n| `scored.json`, `accuracy.csv`, `audit_queue.jsonl` | Answer extraction, correctness, parse failures, truncation, usable coverage, output-token cost, review candidates |\n| `correctness_*.csv`, `answers_*.csv` | Model-by-item correctness and answer matrices per condition |\n| `pairs.csv`, `ladder.csv`, `repeat_pairs.csv` | Double fault, excess overlap, phi/adjusted phi, binary kappa, wrong-answer agreement, one-hot CAPA, relationship labels and question-bootstrap intervals |\n| `teams.csv`, `self_consistency.csv` | Observed all-wrong rate, selection ceiling, correctness majority, actual answer plurality with fractional tie credit, sampled self-consistency and measured token use |\n| `baseline_predictions.csv` | Item-held-out independence and tetrachoric-calibrated shared-factor copula predictions; effective-voter diagnostic |\n| `design_audit.json`, optional `posterior_*.nc` and CSV/JSON | Identifiability checks, Bayesian posterior, variance components, avoidable share, latent ICCs, R-hat/ESS/divergences |\n| Optional `heldout_predictions.csv` | Family/lineage-held-out predictions on held-out items; unseen ability drawn from population prior; independence/copula comparisons |\n| Optional verification reports | Raw accept/reject verdicts, false rejects, false accepts, beta-plus-point-mass fits, and first-three-gate extrapolation through deeper gates |\n| PNG figures, `manifest.json` | Relationship and prediction plots; SHA-256 checksums of saved run files |\n\nBatch elapsed time is shared across responses in that batch. It is not a per-item latency measurement; do not sum it across every response. Token cost is measured, but FLOPs, energy, monetary price, and end-to-end per-item latency are not inferred from tokens. The effective-voter statistic is a variance heuristic, not a predictor of the all-wrong tail.'),
    'Scoring and missingness': mo.md('MC responses must end with `FINAL: A` (a valid option letter). Numeric answers must use a numeric final line. An absent/ambiguous final answer is a parsing failure, not an invented wrong-option answer. Truncated outputs are excluded from shared-failure analysis by default. The conservative pipeline accuracy separately treats parsing/truncation failures as failures. Pair/team reports include their usable item counts; inspect coverage, since complete-case estimates can be biased by nonrandom missingness. Manually adjudicate an extraction sample before production analysis. “Exact” free-response items are case-sensitive string matches; semantic equivalence is not assumed.\n\nThe one-hot CAPA implementation follows equations 2-4 and the explicit deterministic fallback in [Goel et al.](https://arxiv.org/html/2502.04313v1). It does **not** claim to collect option-probability vectors or reproduce soft-probability CAPA. Wrong-answer chance baselines are used only for MC items with known option counts.'),
    'Before Bayesian interpretation': mo.md("Do not invent lineage assignments from checkpoint names. The PDF itself marks several ancestry claims as needing verification. Most catalog lineages intentionally remain unknown. `lineage_verified=True` means you have reviewed a cited source; the supplied verified R1 examples are documented by the [DeepSeek model card](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B). In particular, its 7B Qwen student starts from Qwen2.5-**Math**-7B.\n\nFor each measured model, audit `family`, `lineage`, `base`, `teacher`, and `provenance_url`. Preserve evidence and scope: a mixture of synthetic sources cannot automatically be equated with a pure single teacher. A single categorical lineage is the research plan's simplifying assumption; multi-parent ancestry needs a sensitivity analysis or a weighted-membership model. Do not label all unknown models as one shared lineage. Models with unrelated private post-training recipes also need an explicit evidence-based grouping policy.\n\nThe design audit checks family counts, cross-family lineages, and rank of the pairwise covariance design. It is a necessary check, not proof of statistical identifiability. SmolLM and gpt-oss currently have singleton entries. More eligible checkpoints or a prespecified exclusion are needed if every fitted family must have two models. Changing names alone cannot resolve this.\n\nSet `simulation_recovery=True` to generate/fix a recovery dataset with the selected audited planned roster, then inspect `simulation_truth.json` and sampler diagnostics. This currently runs as an analysis stage after collection/demo: use demo mode with the planned selected roster audited before spending GPU time. One synthetic fit is a plumbing check; repeat across seeds, effect sizes, and item counts before choosing the final design.\n\nSet `fit_bayesian=True` only after pooling and passing the ancestry audit. The default is four chains, 1,000 tuning + 1,000 retained draws each, non-centered item/family/lineage/model effects, and explicit divergent-transition, ESS, and R-hat reports. Set `heldout=True` for the expensive refit grid. Convergence warnings require action; the notebook does not transform a bad posterior into valid scientific evidence. New-model predictions use a population ability prior without test labels. A separate capability-calibrated holdout protocol would answer a different question and must be labeled accordingly."),
    'Optional verification': mo.md('Set `verification=True`, populate `verifier_models` from the catalog, retain `mode="infer"`, and use at least eight sampled repeats for extrapolation to eight gates. Each generator\'s own parsed, untruncated wrong answers form the false-accept population; correct answers are retained as false-reject controls. The verifier never receives the gold label. Exact `ACCEPT`/`REJECT` responses are required.\n\nThe beta-plus-point-mass estimate is exploratory: finite repeated verdicts cannot establish a literal mass of errors that are never caught. First-three-gate fitting is weakly identified; compare sensitivity, bootstrap uncertainty, and empirical deep-gate survival. Current extrapolation reports wrong-answer survival rather than claiming the plan\'s perfect-correct-acceptance reliability formula holds for real verifiers.'),
    'Remaining research work, explicitly outside automatic coverage': mo.md('This is a collection/analysis notebook, not completion of the entire proposed paper. Built-in dataset adapters cover MMLU-Pro, GSM8K, and GPQA-Diamond MC. The following need additional data or protocol work:\n\n- GPQA free response needs audited semantic/reference scoring; MATH symbolic equivalence needs an independently validated scorer; HumanEval+/MBPP+ need an isolated executable-test service. These are not approximated using last-number regexes or unsandboxed generated-code execution.\n- Custom externally scored exact/numeric/MC items can be supplied as JSONL with `id`, `benchmark`, `question`, `format`, `options`, and `gold`, through `custom_items`. Include explicit licenses and scorer provenance in your protocol notes. This does not substitute for execution-based code scoring.\n- Contamination searches against actual training corpora, label-noise adjudication, answer-extraction review, the public vote-corpus pilot and its license check, preregistration, generation/size matching, and measured-compute matching require real evidence. Fully open weights alone do not establish uncontaminated evaluation.\n- The notebook provides pairwise summaries but does not fit a learned pairwise/CAPA team-tail baseline. Add and preregister that model before claiming to beat every baseline in the plan. Soft CAPA and multi-parent lineage are not implemented.\n- Paper-ready variance-component figures, held-out prediction uncertainty beyond reported observed-rate bootstraps, repeated simulation recovery, formal hypothesis tests/multiple-comparison control, blind-spot mixture uncertainty, executable verifier controls, and corpus-wide contamination audits remain necessary for the final study.\n\nThese boundaries are recorded so an unimplemented experiment is not mistaken for a completed one. Core collection is configurable across families, but a guarantee of error-free GPU execution for all candidates would require running them on the actual molab account, with model access and its installed drivers.\n\nImplementation references: [molab documentation](https://docs.marimo.io/guides/molab/), [marimo package management](https://docs.marimo.io/guides/package_management/), [vLLM GPU installation](https://docs.vllm.ai/en/latest/getting_started/installation/gpu/), [MMLU-Pro dataset](https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro). The notebook records resolved runtime versions; it does not ship a fabricated Blackwell-tested environment lock.'),
})

In [ ]:
import ast
import os
import sys
import re
import math
import time
import random
import hashlib
import itertools
import subprocess
import platform
import importlib.metadata
import shutil
from collections import Counter
from decimal import Decimal, InvalidOperation
import numpy as np
import pandas as pd

In [ ]:
def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=lambda x:x.item() if isinstance(x,np.generic) else str(x)).encode()).hexdigest()

In [ ]:
def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    tmp.replace(path)

In [ ]:
def append_record(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())

In [ ]:
def read_records(path):
    if not Path(path).exists():
        return []
    contents = Path(path).read_text(encoding='utf-8')
    lines = contents.splitlines()
    result = []
    for index, line in enumerate(lines):
        try:
            result.append(json.loads(line))
        except json.JSONDecodeError:
            if index != len(lines)-1:
                raise
            # Repair only an interrupted final append, preserving complete rows.
            Path(path).write_text(''.join(json.dumps(r)+'\n' for r in result), encoding='utf-8')
    if contents and not contents.endswith('\n') and len(result)==len(lines):
        with open(path,'a',encoding='utf-8') as f:f.write('\n')
    return result

In [ ]:
def parse_answer(text, item):
    # Only an explicit final answer counts; reasoning letters/numbers are ambiguous.
    clean = text.split('</think>')[-1]
    matches = re.findall(r'(?im)^\s*FINAL\s*:\s*(.+?)\s*$', clean)
    if not matches:
        return None, 'missing_final'
    value = matches[-1].strip()
    if item['format']=='mc':
        m = re.fullmatch(r'\(?([A-Z])\)?[.)]?', value)
        if not m or ord(m[1])-65 >= len(item['options']):
            return None, 'invalid_option'
        return m[1], 'ok'
    if item['format']=='numeric':
        try:
            number = Decimal(value.replace(',','').replace('$','').strip())
            if not number.is_finite():
                return None, 'nonfinite'
            return str(number.normalize()), 'ok'
        except InvalidOperation:
            return None, 'invalid_numeric'
    return value, 'ok'

In [ ]:
def prompt_for(item, template):
    suffix = 'Think through the problem before your final answer.' if template=='reasoned' else 'Answer the question.'
    if item['format']=='mc':
        choices = '\n'.join(chr(65+i)+'. '+str(o) for i,o in enumerate(item['options']))
        return item['question']+'\n'+choices+'\n'+suffix+'\nEnd with exactly one line: FINAL: <option letter>'
    return item['question']+'\n'+suffix+'\nEnd with exactly one line: FINAL: <answer>'

In [ ]:
def validate_items(items):
    if not items or len({x['id'] for x in items})!=len(items):
        raise ValueError('Items must be nonempty with unique stable IDs.')
    for x in items:
        if x['format'] not in ['mc','numeric','exact']:
            raise ValueError('Unsupported scorer: use externally audited exact/numeric/MC items; code is not executed here.')
        if not x.get('question') or not x.get('benchmark'):
            raise ValueError('Missing question/benchmark.')
        if x['format']=='mc' and (not 2<=len(x['options'])<=26 or x['gold'] not in [chr(65+i) for i in range(len(x['options']))]):
            raise ValueError('Invalid MC options or gold.')
        if x['format']=='numeric':
            answer,status=parse_answer('FINAL: '+str(x['gold']),x)
            if status!='ok':raise ValueError('Invalid numeric gold: '+x['id'])
            x['gold']=answer
    return items

In [ ]:
def prepare_items(cfg, root):
    from datasets import load_dataset
    from huggingface_hub import HfApi
    if cfg['custom_items']:
        items = [json.loads(s) for s in Path(cfg['custom_items']).read_text(encoding='utf-8').splitlines() if s.strip()]
        return validate_items(items), {'custom_sha256':digest(items)}
    items, sources = [], {}
    for name in cfg['benchmarks']:
        repo = {'mmlu_pro':'TIGER-Lab/MMLU-Pro','gsm8k':'openai/gsm8k','gpqa':'Idavidrein/gpqa'}[name]
        sha = HfApi().dataset_info(repo, revision=cfg['dataset_revision']).sha
        dsconfig = 'main' if name=='gsm8k' else cfg['gpqa_config'] if name=='gpqa' else None
        split = 'train' if name=='gpqa' else 'test'
        ds = load_dataset(repo, dsconfig, split=split, revision=sha)
        # A fixed shuffle chooses the same questions for every model and setting.
        ds = ds.shuffle(seed=1729)
        if cfg['limit'] is not None:
            ds = ds.select(range(min(cfg['limit'], len(ds))))
        sources[name] = dict(repo=repo, revision=sha, config=dsconfig, split=split, fingerprint=ds._fingerprint)
        for row in ds:
            if name=='mmlu_pro':
                item=dict(question=row['question'], options=row['options'], gold=chr(65+int(row['answer_index'])), format='mc', category=row.get('category',''))
            elif name=='gsm8k':
                gold=row['answer'].split('####')[-1].strip()
                item=dict(question=row['question'], options=[], gold=gold, format='numeric', category='math')
                item['gold']=parse_answer('FINAL: '+gold,item)[0]
            else:
                options=[row['Correct Answer'],row['Incorrect Answer 1'],row['Incorrect Answer 2'],row['Incorrect Answer 3']]
                rng=random.Random(int(digest(row['Question'])[:12],16))
                order=list(range(4)); rng.shuffle(order)
                item=dict(question=row['Question'],options=[options[i] for i in order],gold=chr(65+order.index(0)),format='mc',category=row.get('Subdomain',''))
            item['benchmark']=name
            item['id']=name+':'+digest(item)[:24]
            items.append(item)
    return validate_items(items), sources

In [ ]:
WORKER = __import__('textwrap').dedent(r'''
import os, sys, json, time, hashlib, platform, subprocess, shutil
from pathlib import Path
import importlib.metadata
cfg=json.loads(Path(sys.argv[1]).read_text())
os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
def main():
    import torch
    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer
    if not torch.cuda.is_available(): raise RuntimeError('Attach a GPU in molab first.')
    prop=torch.cuda.get_device_properties(0)
    # Actual kernel execution catches wheels that omit this GPU's architecture.
    a=torch.ones((64,64), device='cuda'); b=a@a; torch.cuda.synchronize()
    try:
        smi=subprocess.run(['nvidia-smi'],capture_output=True,text=True,timeout=30).stdout
    except (OSError,subprocess.TimeoutExpired): smi='unavailable'
    meta={'gpu':prop.name,'vram':prop.total_memory,'capability':torch.cuda.get_device_capability(),
          'cuda':torch.version.cuda,'torch':torch.__version__, 'python':sys.version,
          'platform':platform.platform(),'cpu_count':os.cpu_count(),'nvidia_smi':smi,
          'disk_free_bytes':shutil.disk_usage(Path(cfg['result']).parent).free,
          'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,
          'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
          'packages':{d.metadata['Name']:d.version for d in importlib.metadata.distributions()}}
    Path(cfg['environment']).write_text(json.dumps(meta,indent=2))
    m=cfg['model']
    # Dense BF16 estimate includes a 20% allowance for runtime state, not KV cache.
    # Active MoE parameters do NOT determine storage. Quantized models use explicit/native metadata.
    if not m.get('quantization') and not m.get('native_quantization'):
        estimated=m['size_b']*1e9*2*1.2
        if estimated>prop.total_memory*cfg['gpu_memory_utilization']:
            raise RuntimeError('Estimated weights/runtime exceed GPU budget; choose a documented quantized checkpoint or smaller model.')
    kwargs=dict(model=m['id'],revision=m['revision'],tokenizer_revision=m['revision'],
        dtype=m['dtype'],max_model_len=m['max_model_len'],
        gpu_memory_utilization=cfg['gpu_memory_utilization'],tensor_parallel_size=1,
        trust_remote_code=m['trust_remote_code'],seed=17, enforce_eager=True)
    if m['quantization']: kwargs['quantization']=m['quantization']
    kwargs.update(m.get('engine_kwargs',{}))
    llm=LLM(**kwargs)
    tok=llm.get_tokenizer()
    for start in range(0,len(cfg['jobs']),cfg['batch_size']):
        jobs=cfg['jobs'][start:start+cfg['batch_size']]
        prompts=[]
        for j in jobs:
            if m['kind']=='base': p=j['prompt']
            else:
                if not tok.chat_template: raise ValueError('No native chat template for '+m['id'])
                p=tok.apply_chat_template([{'role':'user','content':j['prompt']}],tokenize=False,
                    add_generation_prompt=True,**m['chat_kwargs'])
            n=len(tok.encode(p,add_special_tokens=False))
            if n+m['max_tokens']>m['max_model_len']:
                raise ValueError('Context overflow: increase max_model_len or decrease generation budget; never silently truncate.')
            j['rendered_prompt']=p
            prompts.append(p)
        params=[SamplingParams(temperature=j['temperature'],top_p=1.0,max_tokens=m['max_tokens'],seed=j['seed']) for j in jobs]
        t=time.perf_counter()
        outputs=llm.generate(prompts,params,use_tqdm=False)
        elapsed=time.perf_counter()-t
        with open(cfg['result'],'a',encoding='utf-8') as f:
            for j,o in zip(jobs,outputs):
                z=o.outputs[0]
                j.update(text=z.text,input_tokens=len(o.prompt_token_ids),output_tokens=len(z.token_ids),
                    finish_reason=z.finish_reason,stop_reason=z.stop_reason,
                    batch_elapsed_s=elapsed,batch_size=len(jobs),batch_start=start,
                    model=m['id'],revision=m['revision'],dtype=m['dtype'],quantization=m['quantization'],
                    utc_completed=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
                f.write(json.dumps(j,ensure_ascii=False)+'\n'); f.flush(); os.fsync(f.fileno())
if __name__=='__main__': main()
''')

In [ ]:
def freeze_roster(cfg, root):
    from huggingface_hub import HfApi, hf_hub_download
    ids=cfg['selected'] + (cfg['verifier_models'] if cfg['verification'] else [])
    catalog={m['id']:m for m in cfg['roster']}
    if len(set(ids))!=len(ids):
        ids=list(dict.fromkeys(ids))
    if not ids or any(i not in catalog for i in ids):
        raise ValueError('Every selected/verifier ID needs a roster entry.')
    result=[]
    for mid in ids:
        m=dict(catalog[mid])
        info=HfApi().model_info(mid,revision=m['revision'])
        m['revision']=info.sha
        m['hub_license']=getattr(info.card_data,'license',None)
        m['hub_base_model']=getattr(info.card_data,'base_model',None)
        modelcfg=json.loads(Path(hf_hub_download(mid,'config.json',revision=info.sha)).read_text())
        m['architectures']=modelcfg.get('architectures',[])
        m['model_type']=modelcfg.get('model_type')
        m['native_quantization']=modelcfg.get('quantization_config')
        card=Path(hf_hub_download(mid,'README.md',revision=info.sha)).read_text(encoding='utf-8')
        card_path=root/'model_cards'/(digest(mid)[:16]+'.md')
        card_path.parent.mkdir(exist_ok=True)
        card_path.write_text(card,encoding='utf-8')
        m['card_sha256']=hashlib.sha256(card.encode()).hexdigest()
        if m['max_tokens']>=m['max_model_len']:
            raise ValueError('max_tokens must leave room for input: '+mid)
        if not m['license_reviewed']:
            print('LICENSE REVIEW PENDING:',mid,m['hub_license'],flush=True)
        result.append(m)
    save_json(root/'resolved_roster.json',result)
    return result

In [ ]:
def infer(cfg, root, roster, items, verification_jobs=None):
    root.mkdir(parents=True,exist_ok=True)
    worker=root/'inference_worker.py'; worker.write_text(WORKER,encoding='utf-8')
    allrows=[]
    for m in roster:
        if verification_jobs is None and m['id'] not in cfg['selected']:
            continue
        if verification_jobs is not None and m['id'] not in cfg['verifier_models']:
            continue
        tag=digest(m['id'])[:16]
        out=root/(tag+'.jsonl')
        done=read_records(out); keys={r['job_id'] for r in done}
        jobs=[]
        if verification_jobs is None:
            for item, template, seed, temp in itertools.product(items,cfg['templates'],cfg['seeds'],cfg['temperatures']):
                job=dict(item_id=item['id'],template=template,seed=seed,temperature=temp,prompt=prompt_for(item,template))
                job['job_id']=digest([m['revision'],job])
                if job['job_id'] not in keys: jobs.append(job)
        else:
            for original in verification_jobs:
                job=dict(original); job['job_id']=digest([m['revision'],job])
                if job['job_id'] not in keys: jobs.append(job)
        if jobs:
            spec=dict(model=m,jobs=jobs,result=str(out.resolve()),environment=str((root/(tag+'_environment.json')).resolve()),
                batch_size=cfg['batch_size'],gpu_memory_utilization=cfg['gpu_memory_utilization'])
            specpath=root/(tag+'_jobs.json'); save_json(specpath,spec)
            print('RUN',m['id'],'pending',len(jobs),'log',str(root/(tag+'.log')),flush=True)
            try:
                with open(root/(tag+'.log'),'a',encoding='utf-8') as log:
                    subprocess.run([sys.executable,str(worker.resolve()),str(specpath.resolve())],stdout=log,stderr=subprocess.STDOUT,
                                   check=True,timeout=cfg['worker_timeout'])
            except (subprocess.CalledProcessError,subprocess.TimeoutExpired) as error:
                tail=(root/(tag+'.log')).read_text(encoding='utf-8',errors='replace')[-6000:]
                raise RuntimeError('Inference stopped; completed rows are resumable. Log: '+str(root/(tag+'.log'))+'\n'+tail) from error
        allrows.extend(read_records(out))
    if verification_jobs is not None:
        return allrows
    byid={x['id']:x for x in items}
    for row in allrows:
        item=byid[row['item_id']]
        row['answer'],row['parse_status']=parse_answer(row['text'],item)
        row['truncated']=row['finish_reason']=='length'
        row['correct']=None if row['answer'] is None else int(row['answer']==item['gold'])
        row['benchmark']=item['benchmark'];row['format']=item['format']
    save_json(root/'scored.json',allrows)
    return allrows

In [ ]:
def synthetic(cfg):
    rng=np.random.default_rng(17)
    roster=[]
    for f in range(15):
        for l in range(2):
            roster.append(dict(id=f'demo/F{f}-M{l}',family=f'F{f}',lineage=f'L{(f+l)%3}',lineage_verified=True,
                size_b=7+l,kind='instruct',base=None,teacher=None))
    items=[dict(id=f'demo:{i}',question=f'Synthetic question {i}',benchmark='synthetic',format='mc',options=['a','b','c','d'],gold='A',category=str(i%4)) for i in range(120)]
    u=rng.normal(0,1.1,len(items));v=rng.normal(0,0.7,(15,len(items)));r=rng.normal(0,0.5,(3,len(items)))
    rows=[]
    for k,m in enumerate(roster):
        p=1/(1+np.exp(-(0.8+0.2*(k%2)+u+v[k//2]+r[int(m['lineage'][1:])])))
        for j,item in enumerate(items):
            answer='A' if rng.random()<p[j] else ['B','C','D'][int(rng.integers(3))]
            rows.append(dict(model=m['id'],item_id=item['id'],benchmark=item['benchmark'],format='mc',
                template='direct',seed=17,temperature=0.0,answer=answer,correct=int(answer=='A'),
                parse_status='ok',truncated=False,text='FINAL: '+answer,output_tokens=3,input_tokens=10,
                finish_reason='stop',job_id=digest([m['id'],item['id']])))
    return roster,items,rows

In [ ]:
def relation(a,b):
    labels=[]
    if a['id']==b['id']: labels.append('same_model')
    if a.get('base')==b['id'] or b.get('base')==a['id']: labels.append('base_derivative')
    if a.get('teacher')==b['id'] or b.get('teacher')==a['id']: labels.append('student_teacher')
    if a.get('base') and a['base']==b.get('base'):
        if (a.get('teacher') and b['kind']=='instruct') or (b.get('teacher') and a['kind']=='instruct'):
            labels.append('student_base_instruct')
    if a['family']==b['family']:
        labels.append('same_family')
        if a.get('size_b')!=b.get('size_b'): labels.append('same_family_different_size')
    known=a.get('lineage') and b.get('lineage')
    if known and a['lineage']==b['lineage']: labels.append('same_lineage')
    if known and a['family']!=b['family'] and a['lineage']!=b['lineage']: labels.append('unrelated')
    return '|'.join(labels) or 'unknown_relationship'

In [ ]:
def pair_metrics(a,b,aa,ab,options):
    ea=1-np.asarray(a,dtype=float);eb=1-np.asarray(b,dtype=float)
    x,y=ea.mean(),eb.mean();df=float((ea*eb).mean());cov=df-x*y
    den=math.sqrt(x*(1-x)*y*(1-y));phi=cov/den if den else np.nan
    low,high=sorted([x,y])
    maximum=math.sqrt(low*(1-high)/(high*(1-low))) if high and low<1 else np.nan
    expected=(1-x)*(1-y)+x*y
    mask=(ea==1)&(eb==1)
    wa=float((np.asarray(aa)[mask]==np.asarray(ab)[mask]).mean()) if mask.any() else np.nan
    chance=float(np.mean(1/(np.asarray(options)[mask]-1))) if mask.any() and np.all(np.asarray(options)[mask]>1) else np.nan
    # Goel et al. 2502.04313 equations 2-4, with one-hot answer distributions.
    # This is not soft-probability CAPA; the paper explicitly permits this reduction.
    capa_chance=(1-x)*(1-y)+x*y*float(np.mean(1/(np.asarray(options)-1))) if np.all(np.asarray(options)>1) else np.nan
    capa=(float((np.asarray(aa)==np.asarray(ab)).mean())-capa_chance)/(1-capa_chance) if np.isfinite(capa_chance) and capa_chance<1 else np.nan
    return dict(n=len(ea),error_a=x,error_b=y,double_fault=df,excess_cofailure=cov,phi=phi,
        adjusted_phi=phi/maximum if maximum and np.isfinite(maximum) else np.nan,
        kappa=(float((ea==eb).mean())-expected)/(1-expected) if expected<1 else np.nan,
        wrong_agreement=wa,wrong_chance_uniform=chance,both_wrong_n=int(mask.sum()),capa_onehot=capa)

In [ ]:
def summarize(cfg,root,roster,items,rows):
    frame=pd.DataFrame(rows)
    valid=frame['correct'].notna() & (cfg['allow_truncated_analysis'] | ~frame['truncated'])
    frame['usable']=valid
    frame['pipeline_correct']=frame['correct'].fillna(0)
    frame.loc[frame['truncated'],'pipeline_correct']=0
    frame['usable_correct']=frame['correct'].where(frame['usable'])
    accuracy=frame.groupby(['model','benchmark','format','template','seed','temperature'],dropna=False).agg(
        n=('item_id','size'),usable=('usable','sum'),pipeline_accuracy=('pipeline_correct','mean'),
        parsed_accuracy=('correct','mean'),parse_failure=('parse_status',lambda s:float((s!='ok').mean())),
        usable_accuracy=('usable_correct','mean'),
        truncation_rate=('truncated','mean'),output_tokens=('output_tokens','sum')).reset_index()
    for prefix,probability,counts in [('pipeline','pipeline_accuracy','n'),('usable','usable_accuracy','usable')]:
        p=accuracy[probability].to_numpy(float);n=accuracy[counts].to_numpy(float);critical=1.959963984540054
        with np.errstate(divide='ignore',invalid='ignore'):
            center=(p+critical**2/(2*n))/(1+critical**2/n)
            half=critical*np.sqrt(p*(1-p)/n+critical**2/(4*n*n))/(1+critical**2/n)
        accuracy[prefix+'_wilson_low']=center-half;accuracy[prefix+'_wilson_high']=center+half
    accuracy.to_csv(root/'accuracy.csv',index=False)
    meta={m['id']:m for m in roster};byitem={i['id']:i for i in items}
    pairs=[];teams=[];ladder=[]
    rng=np.random.default_rng(173)
    # Each condition is separate: no pooling MC with free response or prompt/seed ablations.
    for condition,group in frame.groupby(['benchmark','format','template','seed','temperature'],dropna=False):
        cond=dict(zip(['benchmark','format','template','seed','temperature'],condition))
        wide=group.pivot(index='item_id',columns='model',values='correct')
        usable=group.pivot(index='item_id',columns='model',values='usable')
        answers=group.pivot(index='item_id',columns='model',values='answer')
        wide=wide.where(usable)
        tag=digest(cond)[:12]
        wide.to_csv(root/('correctness_'+tag+'.csv')); answers.to_csv(root/('answers_'+tag+'.csv'))
        pair_arrays=[]
        for a,b in itertools.combinations(wide.columns,2):
            mask=wide[[a,b]].notna().all(axis=1)
            if not mask.any():continue
            z=wide.loc[mask];ans=answers.loc[mask]
            opts=[len(byitem[i]['options']) for i in z.index]
            stats=pair_metrics(z[a],z[b],ans[a],ans[b],opts)
            labels=relation(meta[a],meta[b]);matched=abs(stats['error_a']-stats['error_b'])<=cfg['accuracy_tolerance']
            pairs.append(dict(**cond,a=a,b=b,relationship=labels,accuracy_matched=matched,**stats))
            pair_arrays.append((a,b,labels,matched))
        # Bootstrap a shared draw of QUESTIONS for all pairs in each relationship.
        for label in sorted({s for _,_,ls,_ in pair_arrays for s in ls.split('|')}):
            for matched_only in [False,True]:
                eligible=[(a,b) for a,b,ls,mt in pair_arrays if label in ls.split('|') and (mt or not matched_only)]
                if not eligible:continue
                columns=list(wide.columns)
                left=np.array([columns.index(a) for a,b in eligible]);right=np.array([columns.index(b) for a,b in eligible])
                errors=1-wide.to_numpy(float)
                def estimate(indices):
                    a=errors[indices][:,left];b=errors[indices][:,right]
                    mask=np.isfinite(a)&np.isfinite(b)
                    count=mask.sum(axis=0)
                    with np.errstate(divide='ignore',invalid='ignore'):
                        x=np.where(mask,a,0).sum(axis=0)/count;y=np.where(mask,b,0).sum(axis=0)/count
                        df=np.where(mask,a*b,0).sum(axis=0)/count
                        low=np.minimum(x,y);high=np.maximum(x,y)
                        phi=(df-x*y)/np.sqrt(x*(1-x)*y*(1-y))
                        maximum=np.sqrt(low*(1-high)/(high*(1-low)))
                        values=phi/maximum
                    values=values[np.isfinite(values)]
                    return float(values.mean()) if len(values) else np.nan
                point=estimate(np.arange(len(wide)))
                boots=[estimate(rng.integers(len(wide),size=len(wide))) for _ in range(cfg['bootstrap'])]
                boots=[v for v in boots if np.isfinite(v)]
                ci=np.quantile(boots,[.025,.975]) if boots else [np.nan,np.nan]
                ladder.append(dict(**cond,relationship=label,matched_only=matched_only,pairs=len(eligible),mean_adjusted_phi=point,low=ci[0],high=ci[1]))
        for size in cfg['team_sizes']:
            if size>len(wide.columns):continue
            # Sample combinations without materializing C(K,size).
            total=math.comb(len(wide.columns),size)
            chosen=set()
            if total<=cfg['max_teams']:
                chosen=set(itertools.combinations(wide.columns,size))
            else:
                while len(chosen)<cfg['max_teams']:
                    chosen.add(tuple(sorted(rng.choice(wide.columns,size=size,replace=False))))
            for members in sorted(chosen):
                mask=wide[list(members)].notna().all(axis=1)
                z=wide.loc[mask,list(members)].to_numpy(float)
                if len(z)==0:continue
                ans=answers.loc[mask,list(members)].to_numpy()
                gold=np.array([byitem[i]['gold'] for i in wide.index[mask]])
                beta=(z.sum(axis=1)==0).astype(float)
                majority=(z.sum(axis=1)>size/2).astype(float)
                # Fractional tie credit is deterministic; no arbitrary model-order tiebreak.
                plurality=[]
                for a,g in zip(ans,gold):
                    counts=Counter(a);mx=max(counts.values());tops=[k for k,v in counts.items() if v==mx]
                    plurality.append(float(g in tops)/len(tops))
                ids=rng.integers(len(z),size=(cfg['bootstrap'],len(z)))
                ci=np.quantile(beta[ids].mean(axis=1),[.025,.975])
                teams.append(dict(**cond,team=json.dumps(members),n=len(z),beta=float(beta.mean()),beta_low=ci[0],beta_high=ci[1],
                    selection_ceiling=1-float(beta.mean()),correctness_majority=float(majority.mean()),plurality_accuracy=float(np.mean(plurality)),
                    independence_beta=float(np.prod(1-z.mean(axis=0)))))
    pd.DataFrame(pairs).to_csv(root/'pairs.csv',index=False)
    pd.DataFrame(ladder).to_csv(root/'ladder.csv',index=False)
    pd.DataFrame(teams).to_csv(root/'teams.csv',index=False)
    # Same model repeats and self-consistency need distinct conditions; never fake additional independent models.
    repeat=[];selfvotes=[]
    for (mid,bench,fmt),group in frame.groupby(['model','benchmark','format']):
        group=group[group['usable']]
        conditions=['template','seed','temperature']
        group=group.assign(condition=group[conditions].astype(str).agg('|'.join,axis=1))
        w=group.pivot(index='item_id',columns='condition',values='correct')
        a=group.pivot(index='item_id',columns='condition',values='answer')
        for ca,cb in itertools.combinations(w.columns,2):
            mask=w[[ca,cb]].notna().all(axis=1)
            if mask.any():
                repeat.append(dict(model=mid,benchmark=bench,format=fmt,condition_a=ca,condition_b=cb,
                    **pair_metrics(w.loc[mask,ca],w.loc[mask,cb],a.loc[mask,ca],a.loc[mask,cb],[len(byitem[i]['options']) for i in w.index[mask]])))
        # Self-consistency uses sampled seeds within a single prompt and temperature.
        for (template,temp),sg in group[group.temperature>0].groupby(['template','temperature']):
            aw=sg.pivot(index='item_id',columns='seed',values='answer').dropna()
            for size in cfg['team_sizes']:
                if aw.shape[1]<size:continue
                sub=aw.iloc[:,:size];scores=[]
                for item_id,values in sub.iterrows():
                    counts=Counter(values);mx=max(counts.values());tops=[k for k,v in counts.items() if v==mx]
                    scores.append(float(byitem[item_id]['gold'] in tops)/len(tops))
                if not scores:continue
                seedset=list(sub.columns)
                costs=sg[sg.seed.isin(seedset)&sg.item_id.isin(sub.index)].output_tokens.sum()
                selfvotes.append(dict(model=mid,benchmark=bench,format=fmt,template=template,temperature=temp,
                    samples=size,items=len(scores),plurality_accuracy=float(np.mean(scores)),output_tokens=int(costs),
                    cost_policy='Match ensemble measured tokens, not merely number of calls.'))
    pd.DataFrame(repeat).to_csv(root/'repeat_pairs.csv',index=False)
    pd.DataFrame(selfvotes).to_csv(root/'self_consistency.csv',index=False)
    frame.loc[(~frame['usable']) | (frame['pipeline_correct']==0)].to_json(root/'audit_queue.jsonl',orient='records',lines=True,force_ascii=False)
    return accuracy,frame,pd.DataFrame(teams)

In [ ]:
def audit_design(roster):
    families=Counter(m['family'] for m in roster)
    verified=[m for m in roster if m.get('lineage') and m.get('lineage_verified')]
    lineages={l:{m['family'] for m in verified if m['lineage']==l} for l in {m['lineage'] for m in verified}}
    pairs=list(itertools.combinations(verified,2))
    design=np.array([[1,int(a['family']==b['family']),int(a['lineage']==b['lineage'])] for a,b in pairs])
    rank=int(np.linalg.matrix_rank(design)) if len(design) else 0
    issues=[]
    if len(roster)<20:issues.append('Fewer than 20 models.')
    if len(families)<6:issues.append('Fewer than six families.')
    if any(n<2 for n in families.values()):issues.append('Some families have only one model.')
    if len(verified)!=len(roster):issues.append('Unknown or unaudited lineage entries; do not replace these with one shared unknown group.')
    if any(len(f)<2 for f in lineages.values()):issues.append('Some lineages do not span two families.')
    if rank<3:issues.append('Family/lineage covariance design is rank deficient.')
    return dict(families=dict(families),lineage_families={k:sorted(v) for k,v in lineages.items()},covariance_rank=rank,issues=issues)

In [ ]:
def fit_model(z,roster,cfg,path):
    import pymc as pm
    import arviz as az
    families=sorted({m['family'] for m in roster});lineages=sorted({m['lineage'] for m in roster})
    f=np.array([families.index(m['family']) for m in roster]);l=np.array([lineages.index(m['lineage']) for m in roster])
    ki,xi=np.where(np.isfinite(z))
    with pm.Model() as model:
        mu=pm.Normal('mu',0,2)
        su=pm.HalfNormal('sigma_u',1);sf=pm.HalfNormal('sigma_f',1);sl=pm.HalfNormal('sigma_l',1);sw=pm.HalfNormal('sigma_w',1)
        w=pm.Normal('w_raw',0,1,shape=z.shape[0])*sw
        u=pm.Normal('u_raw',0,1,shape=z.shape[1])*su
        v=pm.Normal('v_raw',0,1,shape=(len(families),z.shape[1]))*sf
        r=pm.Normal('r_raw',0,1,shape=(len(lineages),z.shape[1]))*sl
        pm.Bernoulli('obs',logit_p=mu+w[ki]+u[xi]+v[f[ki],xi]+r[l[ki],xi],observed=z[ki,xi].astype(int))
        pm.Deterministic('avoidable_share',(sf**2+sl**2)/(su**2+sf**2+sl**2))
        for name,s in [('univ',su),('fam',sf),('lin',sl)]:
            pm.Deterministic('ICC_'+name,s**2/(su**2+sf**2+sl**2+np.pi**2/3))
        for name,s in [('u',su),('f',sf),('l',sl),('w',sw)]:pm.Deterministic('variance_'+name,s**2)
        trace=pm.sample(draws=cfg['draws'],tune=cfg['tune'],chains=cfg['chains'],cores=1,
                        target_accept=cfg['target_accept'],random_seed=17,return_inferencedata=True)
    trace.to_netcdf(str(path)+'.nc')
    names=['mu','sigma_u','sigma_f','sigma_l','sigma_w','avoidable_share','ICC_univ','ICC_fam','ICC_lin','variance_u','variance_f','variance_l','variance_w']
    summary=az.summary(trace,var_names=names,hdi_prob=.95)
    summary.to_csv(str(path)+'_summary.csv')
    diagnostic=dict(divergences=int(trace.sample_stats['diverging'].sum()),max_rhat=float(summary['r_hat'].max()),min_ess=float(summary['ess_bulk'].min()))
    save_json(str(path)+'_diagnostics.json',diagnostic)
    if diagnostic['divergences'] or diagnostic['max_rhat']>1.01 or diagnostic['min_ess']<400:
        print('POSTERIOR DIAGNOSTICS REQUIRE REVIEW',diagnostic,flush=True)
    return trace

In [ ]:
def posterior_team(trace,training,team,cfg,rng):
    # Prediction on NEW items. Known ability is inferred from training items only.
    post=trace.posterior.stack(sample=('chain','draw'))
    count=cfg['posterior_simulations'];ix=rng.integers(post.sizes['sample'],size=count)
    values=lambda name: np.asarray(post[name].values)[...,ix]
    mu=values('mu');su=values('sigma_u');sf=values('sigma_f');sl=values('sigma_l');sw=values('sigma_w')
    known={m['id']:i for i,m in enumerate(training)}
    u=rng.normal(size=count)*su
    vf={f:rng.normal(size=count)*sf for f in {m['family'] for m in team}}
    rl={l:rng.normal(size=count)*sl for l in {m['lineage'] for m in team}}
    probabilities=[]
    for m in team:
        w=values('w_raw')[known[m['id']]]*sw if m['id'] in known else rng.normal(size=count)*sw
        eta=mu+w+u+vf[m['family']]+rl[m['lineage']]
        probabilities.append(1/(1+np.exp(-eta)))
    p=np.array(probabilities)
    beta=float(np.prod(1-p,axis=0).mean())
    z=rng.random(p.shape)<p
    return beta,float((z.sum(axis=0)>len(team)/2).mean())

In [ ]:
def run_bayesian(cfg,root,roster,frame,teams):
    roster=[m for m in roster if m['id'] in set(frame['model'])]
    report=audit_design(roster);save_json(root/'design_audit.json',report)
    if report['issues'] and cfg['require_lineage_audit']:
        raise ValueError('Bayesian analysis blocked by design audit: '+' '.join(report['issues']))
    if any(not m.get('lineage') for m in roster):
        raise ValueError('Explicit audited lineage assignment required for every model in this analysis.')
    rng=np.random.default_rng(53)
    output=[]
    for cond,g in frame.groupby(['benchmark','format','template','seed','temperature']):
        z=g.pivot(index='model',columns='item_id',values='correct').where(g.pivot(index='model',columns='item_id',values='usable'))
        selected=[m for m in roster if m['id'] in z.index]
        z=z.reindex([m['id'] for m in selected]);tag=digest(cond)[:12]
        trace=fit_model(z.to_numpy(float),selected,cfg,root/('posterior_'+tag))
        if not cfg['heldout']:continue
        # Items split BEFORE fitting; no test-item correctness enters capability calibration.
        order=rng.permutation(z.shape[1]);cut=z.shape[1]//2
        train_items,test_items=order[:cut],order[cut:]
        for axis in ['family','lineage']:
            for value in sorted({m[axis] for m in selected}):
                keep=[i for i,m in enumerate(selected) if m[axis]!=value]
                held=[i for i,m in enumerate(selected) if m[axis]==value]
                if len(keep)<4:continue
                training=[selected[i] for i in keep]
                training_z=z.to_numpy(float)[np.ix_(keep,train_items)]
                tr=fit_model(training_z,training,cfg,root/('heldout_'+tag+'_'+digest([axis,value])[:10]))
                _,_,foldrho=copula_baseline(training_z,list(range(min(3,len(training)))),rng,100)
                erates=np.nanmean(1-training_z,axis=1);population_error=float(np.nanmean(erates))
                combinations=[]
                for h in held:
                    combinations.extend((h,*rest) for rest in itertools.islice(itertools.combinations(keep,2),cfg['max_teams']//max(len(held),1)))
                for members in combinations:
                    team=[selected[i] for i in members]
                    obs=z.to_numpy(float)[np.ix_(members,test_items)]
                    obs=obs[:,np.isfinite(obs).all(axis=0)]
                    if not obs.shape[1]:continue
                    beta,maj=posterior_team(tr,training,team,cfg,rng)
                    empirical=float((obs.sum(axis=0)==0).mean())
                    majority=float((obs.sum(axis=0)>len(members)/2).mean())
                    from scipy.stats import norm
                    errors=np.array([erates[keep.index(i)] if i in keep else population_error for i in members])
                    independence=float(np.prod(errors))
                    xi=rng.normal(size=cfg['posterior_simulations'])
                    pe=norm.cdf((norm.ppf(np.clip(errors,1e-4,1-1e-4))[:,None]-np.sqrt(foldrho)*xi)/np.sqrt(1-foldrho))
                    copula=float(np.prod(pe,axis=0).mean())
                    output.append(dict(condition=json.dumps(cond,default=str),axis=axis,heldout=value,team=json.dumps([m['id'] for m in team]),
                        n=obs.shape[1],pred_beta=beta,observed_beta=empirical,beta_mae=abs(beta-empirical),pred_majority=maj,observed_majority=majority,
                        independence_beta=independence,independence_beta_mae=abs(independence-empirical),
                        single_factor_beta=copula,single_factor_beta_mae=abs(copula-empirical),
                        majority_mae=abs(maj-majority),ability_policy='unseen model: population prior; known model: training items only'))
    pd.DataFrame(output).to_csv(root/'heldout_predictions.csv',index=False)

In [ ]:
def copula_baseline(z,team,rng,samples):
    # Tetrachoric rho: fit each bivariate normal CDF to the observed double fault.
    from scipy.stats import norm, multivariate_normal
    from scipy.optimize import brentq
    errors=1-z
    rates=np.clip(np.nanmean(errors,axis=1),1e-4,1-1e-4)
    thresholds=norm.ppf(rates)
    rhos=[]
    for a,b in itertools.combinations(range(len(z)),2):
        mask=np.isfinite(errors[a])&np.isfinite(errors[b])
        if mask.sum()<5:continue
        ea=errors[a,mask];eb=errors[b,mask]
        if not 0<ea.mean()<1 or not 0<eb.mean()<1:continue
        ta,tb=norm.ppf([ea.mean(),eb.mean()]);target=float((ea*eb).mean())
        fun=lambda rho: float(multivariate_normal.cdf([ta,tb],mean=[0,0],cov=[[1,rho],[rho,1]],rng=np.random.default_rng(99)))-target
        try:rhos.append(brentq(fun,-.99,.99,xtol=1e-4))
        except ValueError:pass
    rho=float(np.clip(np.mean(rhos),0,.99)) if rhos else 0.0
    xi=rng.normal(size=samples)
    pe=norm.cdf((thresholds[list(team),None]-np.sqrt(rho)*xi)/np.sqrt(1-rho))
    zsim=rng.random(pe.shape)>pe
    return float(np.prod(pe,axis=0).mean()),float((zsim.sum(axis=0)>len(team)/2).mean()),rho

In [ ]:
def baseline_reports(cfg,root,frame,items):
    # All baseline fitting uses a fixed half of items, evaluated on the other half.
    rng=np.random.default_rng(53);results=[]
    for condition,g in frame.groupby(['benchmark','format','template','seed','temperature']):
        wide=g.pivot(index='model',columns='item_id',values='correct').where(g.pivot(index='model',columns='item_id',values='usable'))
        z=wide.to_numpy(float);order=rng.permutation(z.shape[1]);cut=z.shape[1]//2
        if cut<3 or z.shape[0]<3:continue
        train=z[:,order[:cut]];test=z[:,order[cut:]]
        # Fit shared-factor calibration once on the training pool, never on test answers.
        _,_,rho=copula_baseline(train,[0,1,2],rng,100)
        from scipy.stats import norm
        e=np.clip(np.nanmean(1-train,axis=1),1e-4,1-1e-4)
        eligible=list(itertools.islice(itertools.combinations(range(len(z)),3),cfg['max_teams']))
        for team in eligible:
            obs=test[list(team)];obs=obs[:,np.isfinite(obs).all(axis=0)]
            if obs.shape[1]==0:continue
            xi=rng.normal(size=cfg['posterior_simulations'])
            pe=norm.cdf((norm.ppf(e[list(team)])[:,None]-np.sqrt(rho)*xi)/np.sqrt(1-rho))
            error=(obs.sum(axis=0)==0).astype(float)
            majority=float((obs.sum(axis=0)>len(team)/2).mean())
            corr=np.corrcoef(train[list(team)])
            avg=float(np.nanmean(corr[np.triu_indices(len(team),1)]))
            effective=len(team)/(1+(len(team)-1)*avg) if 1+(len(team)-1)*avg>0 else np.nan
            # Effective voters is a variance heuristic; it cannot identify the all-wrong tail.
            ids=rng.integers(len(error),size=(cfg['bootstrap'],len(error)))
            ci=np.quantile(error[ids].mean(axis=1),[.025,.975])
            predictions={'independence':float(np.prod(e[list(team)])),'single_factor_tetrachoric':float(np.prod(pe,axis=0).mean())}
            p=1-e[list(team)]
            distribution=np.array([1.0])
            for prob in p:distribution=np.convolve(distribution,[1-prob,prob])
            independent_majority=float(distribution[len(team)//2+1:].sum())
            cp=1-pe
            maj_sim=(rng.random(cp.shape)<cp).sum(axis=0)>len(team)/2
            for method,pred in predictions.items():
                predmajor=independent_majority if method=='independence' else float(maj_sim.mean())
                results.append(dict(condition=json.dumps(condition,default=str),team=json.dumps([wide.index[i] for i in team]),method=method,
                    train_items=cut,test_items=len(error),observed_beta=float(error.mean()),beta_low=ci[0],beta_high=ci[1],predicted_beta=pred,
                    beta_mae=abs(pred-error.mean()),observed_majority=majority,predicted_majority=predmajor,majority_mae=abs(predmajor-majority),
                    copula_rho=rho,effective_voters=effective,policy='item holdout; capabilities calibrated on training items'))
    pd.DataFrame(results).to_csv(root/'baseline_predictions.csv',index=False)

In [ ]:
def recovery_check(cfg,root):
    # Fits the exact generating model, retaining the real planned family/lineage crossing.
    roster=[m for m in cfg['roster'] if m['id'] in cfg['selected']]
    if any(not m.get('lineage') or not m.get('lineage_verified') for m in roster):
        raise ValueError('Simulation recovery requires the audited planned roster.')
    rng=np.random.default_rng(721);n=cfg['limit'] or 1000
    families=sorted({m['family'] for m in roster});lineages=sorted({m['lineage'] for m in roster})
    truth=dict(sigma_u=1.1,sigma_f=.7,sigma_l=.5,sigma_w=.4)
    u=rng.normal(0,truth['sigma_u'],n);v=rng.normal(0,truth['sigma_f'],(len(families),n));r=rng.normal(0,truth['sigma_l'],(len(lineages),n))
    w=rng.normal(0,truth['sigma_w'],len(roster))
    eta=np.array([.8+w[k]+u+v[families.index(m['family'])]+r[lineages.index(m['lineage'])] for k,m in enumerate(roster)])
    p=1/(1+np.exp(-eta));z=(rng.random(p.shape)<p).astype(float)
    trace=fit_model(z,roster,cfg,root/'simulation_recovery')
    result=[]
    for name,true in truth.items():
        a=np.asarray(trace.posterior[name]).flatten();lo,hi=np.quantile(a,[.025,.975])
        result.append(dict(parameter=name,truth=true,mean=float(a.mean()),low=float(lo),high=float(hi),truth_in_interval=bool(lo<=true<=hi)))
    save_json(root/'simulation_truth.json',result)

In [ ]:
def merge_runs(cfg):
    roster={};items=None;rows={};reference=None
    for directory in cfg['merge_roots']:
        root=Path(directory)
        c=json.loads((root/'config.json').read_text())
        fields=['benchmarks','limit','dataset_revision','custom_items','templates','seeds','temperatures']
        protocol={k:c[k] for k in fields}
        if reference is None:reference=protocol
        if protocol!=reference:raise ValueError('Cannot pool incompatible experiment protocols.')
        ii=json.loads((root/'items.json').read_text())
        if items is None:items=ii
        if digest(items)!=digest(ii):raise ValueError('Cannot pool runs with different item IDs, labels or ordering.')
        for m in json.loads((root/'resolved_roster.json').read_text()):
            if m['id'] in roster and digest(roster[m['id']])!=digest(m):raise ValueError('Conflicting frozen roster metadata for '+m['id'])
            roster[m['id']]=m
        for row in json.loads((root/'scored.json').read_text()):
            key=(row['model'],row['item_id'],row['template'],row['seed'],row['temperature'])
            if key in rows and digest(rows[key])!=digest(row):raise ValueError('Duplicate response conflict; explicitly choose one run.')
            rows[key]=row
    return list(roster.values()),items,list(rows.values())

In [ ]:
def make_figures(root):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ladder=pd.read_csv(root/'ladder.csv')
    if len(ladder):
        # Separate panels per experiment condition, no hidden pooling of prompts or tasks.
        for condition,g in ladder.groupby(['benchmark','format','template','seed','temperature']):
            g=g[~g.matched_only].dropna(subset=['mean_adjusted_phi'])
            if g.empty:continue
            fig,ax=plt.subplots(figsize=(9,5))
            x=np.arange(len(g));y=g.mean_adjusted_phi.to_numpy()
            ax.errorbar(x,y,yerr=np.maximum(0,np.array([y-g.low.to_numpy(),g.high.to_numpy()-y])),fmt='o',capsize=4)
            ax.set_xticks(x,g.relationship,rotation=30,ha='right');ax.set_ylabel('Mean adjusted phi (question bootstrap 95% interval)')
            ax.set_title(' / '.join(map(str,condition)));fig.tight_layout();fig.savefig(root/('ladder_'+digest(condition)[:12]+'.png'),dpi=160);plt.close(fig)
    for filename in ['baseline_predictions.csv','heldout_predictions.csv']:
        path=root/filename
        if not path.exists() or path.stat().st_size<5:continue
        frame=pd.read_csv(path)
        predicted='pred_beta' if 'pred_beta' in frame else 'predicted_beta'
        if frame.empty:continue
        fig,ax=plt.subplots(figsize=(6,6))
        if 'method' in frame:
            for method,g in frame.groupby('method'):ax.scatter(g.observed_beta,g[predicted],label=method,s=15,alpha=.6)
            ax.legend()
        else:ax.scatter(frame.observed_beta,frame[predicted],s=15,alpha=.6)
        top=max(float(frame.observed_beta.max()),float(frame[predicted].max()),.01)
        ax.plot([0,top],[0,top],color='gray',linestyle='--');ax.set_xlabel('Observed all-wrong rate');ax.set_ylabel('Predicted all-wrong rate')
        fig.tight_layout();fig.savefig(root/(path.stem+'.png'),dpi=160);plt.close(fig)

In [ ]:
def verifier_analysis(cfg,root,verdicts):
    from scipy.optimize import minimize
    from scipy.special import betaln,gammaln,expit
    frame=pd.DataFrame(verdicts)
    frame['accept']=frame['text'].str.strip().str.fullmatch('ACCEPT').astype(int)
    frame['valid']=frame['text'].str.strip().isin(['ACCEPT','REJECT']) & (frame['finish_reason']!='length')
    frame.to_json(root/'verifier_verdicts.jsonl',orient='records',lines=True)
    output=[];curves=[]
    for (generator,verifier),g in frame.groupby(['generator','model']):
        control=g[g['source_correct']==1]
        false_reject=1-float(control.loc[control['valid'],'accept'].mean())
        g=g[g['source_correct']==0]
        counts=g.groupby('source_job').agg(x=('accept','sum'),n=('valid','size'),valid=('valid','all'))
        counts=counts[counts['valid'] & (counts['n']==cfg['verifier_repeats'])]
        if len(counts)==0:continue
        x=counts.x.to_numpy();n=counts.n.to_numpy();mean=float(np.mean(x/n))
        m2=float(np.mean(x*(x-1)/(n*(n-1))))
        rho=(m2-mean**2)/(mean*(1-mean)) if 0<mean<1 else None
        def objective(theta):
            a,b=np.exp(theta[:2]);mass=expit(theta[2])
            logbb=gammaln(n+1)-gammaln(x+1)-gammaln(n-x+1)+betaln(x+a,n-x+b)-betaln(a,b)
            logmix=np.logaddexp(np.log1p(-mass)+logbb,np.where(x==n,np.log(mass),-np.inf))
            return -float(logmix.sum())
        fits=[minimize(objective,[a,b,c],method='L-BFGS-B',bounds=[(-6,6),(-6,6),(-12,12)]) for a,b,c in [(0,0,-2),(1,1,-5),(-1,1,0)]]
        fit=min(fits,key=lambda f:f.fun);a,b=np.exp(fit.x[:2]);mass=float(expit(fit.x[2]))
        # Low-order fit uses only the first three gates, with deeper gates held out.
        ordered=g.sort_values('seed');first=ordered.groupby('source_job').head(3)
        lowcounts=first.groupby('source_job').agg(x=('accept','sum'),n=('valid','size'),valid=('valid','all'))
        lowcounts=lowcounts[lowcounts.valid & (lowcounts.n==3)]
        if len(lowcounts):
            oldx,oldn=x,n
            x=lowcounts.x.to_numpy();n=lowcounts.n.to_numpy()
            lowfits=[minimize(objective,[0,0,c],method='L-BFGS-B',bounds=[(-6,6),(-6,6),(-12,12)]) for c in [-5,-2,0]]
            lowfit=min(lowfits,key=lambda f:f.fun);la,lb=np.exp(lowfit.x[:2]);lm=float(expit(lowfit.x[2]))
            x,n=oldx,oldn
            for depth in range(1,cfg['verifier_repeats']+1):
                subset=ordered.groupby('source_job').head(depth)
                groups=subset.groupby('source_job').agg(accept_all=('accept','prod'),valid=('valid','all'),n=('valid','size'))
                groups=groups[groups.valid & (groups.n==depth)]
                empirical=float(groups.accept_all.mean())
                predicted=float(lm+(1-lm)*np.exp(betaln(la+depth,lb)-betaln(la,lb)))
                curves.append(dict(generator=generator,verifier=verifier,depth=depth,fit_depth=3,
                    observed_wrong_survival=empirical,predicted_wrong_survival=predicted,independent_wrong_survival=mean**depth,
                    heldout_depth=depth>3,warning='Three gates weakly identify a three-parameter mixture; inspect sensitivity.'))
        output.append(dict(generator=generator,verifier=verifier,items=len(x),mean_false_accept=mean,m2=m2,rho=rho,
            beta_a=a,beta_b=b,blindspot_mass=mass,optimizer_success=bool(fit.success),false_reject_rate=false_reject,
            caveat='Finite-repeat mixture estimate; uncertainty and non-identifiability audit required; not proof of never-caught errors.'))
    pd.DataFrame(output).to_csv(root/'verification_summary.csv',index=False)
    pd.DataFrame(curves).to_csv(root/'verification_extrapolation.csv',index=False)

In [ ]:
def run_experiment(cfg):
    cfg=json.loads(json.dumps(cfg))
    if cfg['mode'] not in ['demo','infer','analyze']:raise ValueError('Unknown mode')
    if cfg['bootstrap']<1 or not cfg['templates'] or not cfg['seeds'] or not cfg['temperatures']:raise ValueError('Invalid experiment grid')
    base=Path(cfg['output_root']).expanduser().resolve();base.mkdir(parents=True,exist_ok=True)
    # Analysis switches do not create a new inference experiment or invalidate checkpoints.
    infer_cfg={k:v for k,v in cfg.items() if k not in ['mode','output_root','bootstrap','accuracy_tolerance','team_sizes','max_teams','fit_bayesian','heldout','draws','tune','chains','target_accept','posterior_simulations','require_lineage_audit','allow_truncated_analysis','simulation_recovery']}
    infer_cfg['worker_sha256']=hashlib.sha256(WORKER.encode()).hexdigest()
    infer_cfg['protocol_schema']=1
    key=('demo_' if cfg['mode']=='demo' else 'experiment_')+digest(infer_cfg)[:16]
    if cfg['mode']=='analyze' and cfg['merge_roots']:key='pooled_'+digest(cfg['merge_roots'])[:16]
    root=base/key;root.mkdir(exist_ok=True)
    # Exclusive lock prevents two reactive executions writing the same run.
    lock=root/'RUNNING.lock'
    try:
        fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY);os.write(fd,str(os.getpid()).encode());os.close(fd)
    except FileExistsError:
        raise RuntimeError('Another run or interrupted session owns '+str(lock)+'. Confirm it stopped before removing the lock.')
    gpu_lock=base/'GPU_RUNNING.lock';gpu_owned=False
    try:
        if cfg['mode']=='infer':
            try:
                gpu_fd=os.open(gpu_lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
                os.write(gpu_fd,str(os.getpid()).encode());os.close(gpu_fd);gpu_owned=True
            except FileExistsError:
                raise RuntimeError('A GPU run already owns this output root. Confirm it has stopped before removing '+str(gpu_lock))
        if cfg['mode']=='demo':
            roster,items,rows=synthetic(cfg)
            save_json(root/'resolved_roster.json',roster);save_json(root/'items.json',items);save_json(root/'scored.json',rows)
        elif cfg['mode']=='analyze' and cfg['merge_roots']:
            roster,items,rows=merge_runs(cfg)
            save_json(root/'resolved_roster.json',roster);save_json(root/'items.json',items);save_json(root/'scored.json',rows)
        elif cfg['mode']=='analyze':
            for name in ['resolved_roster.json','items.json','scored.json']:
                if not (root/name).exists():raise ValueError('No matching completed inference run at '+str(root))
            roster=json.loads((root/'resolved_roster.json').read_text());items=json.loads((root/'items.json').read_text());rows=json.loads((root/'scored.json').read_text())
        else:
            if platform.system()!='Linux':raise RuntimeError('Inference requires Linux molab with GPU; use demo locally.')
            # Freeze only once; reruns never silently switch model or dataset revisions.
            if (root/'resolved_roster.json').exists():
                roster=json.loads((root/'resolved_roster.json').read_text())
            else:roster=freeze_roster(cfg,root)
            if (root/'items.json').exists():items=json.loads((root/'items.json').read_text())
            else:
                items,sources=prepare_items(cfg,root);save_json(root/'items.json',items);save_json(root/'datasets.json',sources)
            rows=infer(cfg,root,roster,items)
        save_json(root/'config.json',cfg)
        save_json(root/'design_audit.json',audit_design(roster))
        accuracy,frame,teams=summarize(cfg,root,roster,items,rows)
        baseline_reports(cfg,root,frame,items)
        if cfg['simulation_recovery']:recovery_check(cfg,root)
        if cfg['fit_bayesian']:run_bayesian(cfg,root,roster,frame,teams)
        if cfg['verification']:
            if cfg['mode']!='infer':raise ValueError('Verification requires infer mode and real saved generations.')
            if cfg['verifier_repeats']<2 or not cfg['verifier_models']:raise ValueError('Specify verifiers and at least two repeats.')
            jobs=[]
            # Include correct controls as well as own errors; false rejects remain measurable.
            for row in rows:
                if row['parse_status']!='ok' or row['truncated']:continue
                item=next(i for i in items if i['id']==row['item_id'])
                for repeat in range(cfg['verifier_repeats']):
                    jobs.append(dict(source_job=row['job_id'],generator=row['model'],source_correct=row['correct'],
                        item_id=row['item_id'],template='verify',seed=1000+repeat,temperature=cfg['verifier_temperature'],
                        prompt='Check the following candidate answer. Reply with exactly ACCEPT or REJECT.\n'+prompt_for(item,'direct')+'\nCandidate:\n'+row['text']))
            if not jobs:raise ValueError('No parsed, untruncated generator answers are available for verification; inspect the audit queue first.')
            vr=infer(cfg,root/'verification',roster,items,jobs)
            verifier_analysis(cfg,root,vr)
        make_figures(root)
        manifest={str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in root.rglob('*') if p.is_file() and p!=lock and p.name!='manifest.json'}
        save_json(root/'manifest.json',manifest)
        summary=f"Completed **{cfg['mode']}**: {len(rows):,} responses, {len({r['model'] for r in rows})} models. Results: `{root}`.\n\n"
        summary+='Synthetic results validate plumbing only.' if cfg['mode']=='demo' else 'Inspect audit_queue.jsonl, truncation, parsing coverage, metadata, and posterior diagnostics before interpretation.'
        return dict(summary=summary,accuracy=accuracy.to_dict('records'),directory=str(root))
    finally:
        lock.unlink(missing_ok=True)
        if gpu_owned:gpu_lock.unlink(missing_ok=True)

In [ ]:
# Quick local checks execute before the expensive inference cell.
ast.parse(WORKER)
assert parse_answer("FINAL: A", {"format": "mc", "options": ["x", "y"]}) == ("A", "ok")
assert parse_answer("reasoning mentions A", {"format": "mc", "options": ["x", "y"]})[0] is None
assert parse_answer("FINAL: 42.00", {"format": "numeric"}) == ("42", "ok")
_metric = pair_metrics([1, 1, 0, 0], [1, 0, 1, 0], list("ABCD"), list("ACBD"), [4]*4)
assert abs(_metric["phi"]) < 1e-12
CHECKS_PASSED = True
mo.md("Parsing, pair-metric and worker-syntax checks passed. The next cell runs the configured experiment.")

In [ ]:
import sys as _setup_sys
import shutil as _setup_shutil
import subprocess as _setup_subprocess

print("Installing into worker Python:", _setup_sys.executable)

_setup_uv = _setup_shutil.which("uv")
if _setup_uv is None:
    raise RuntimeError(
        "uv was not found. Install vllm through molab's Packages panel, "
        "then rerun this check."
    )

_setup_subprocess.run(
    [
        _setup_uv, "pip", "install",
        "--python", _setup_sys.executable,
        "vllm",
        "--torch-backend=auto",
    ],
    check=True,
)

# Test in a fresh process, just as the inference worker runs.
_setup_subprocess.run(
    [
        _setup_sys.executable,
        "-c",
        """
import torch
import vllm
from vllm import LLM, SamplingParams

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "CUDA GPU is unavailable"
x = torch.ones((32, 32), device="cuda")
y = x @ x
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("PASS: worker imports and basic CUDA operation")
""",
    ],
    check=True,
)

In [ ]:
assert CHECKS_PASSED
RESULT = run_experiment(CONFIG)

In [ ]:
mo.vstack([mo.md(RESULT["summary"]), mo.ui.table(RESULT["accuracy"], pagination=True)])

In [ ]:
from pathlib import Path as _export_Path
import shutil as _export_shutil
import marimo as _export_mo

_export_folder = _export_Path(
    "/marimo/lineage_results/experiment_e65f0d9272abef25"
)

_export_zip = _export_shutil.make_archive(
    base_name=str(_export_folder.parent / "yi_results_e65f0d9272abef25"),
    format="zip",
    root_dir=str(_export_folder.parent),
    base_dir=_export_folder.name,
)

_export_mo.download(
    data=_export_Path(_export_zip).read_bytes(),
    filename="yi_results_e65f0d9272abef25.zip",
    mimetype="application/zip",
    label="Download all Yi results",
)